# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruzaki11/Flyrank-ml-intern-tasks/blob/main/work/notebooks/w04_baseline_score.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [1]:
import pandas as pd

df = pd.read_csv("hf://datasets/FlyRank/internship-starter/content_refresh_anonymized.csv")

In [2]:
label = "is_initial_refresh_candidate"

I will adress the signals first then create a rule :

-The First Signal is : Staleness

-Verdict : Mixed

Refresh-candidate rates increase for moderately stale pages, but the oldest buckets contain very few observations and do not continue the trend

In [3]:
bins = [0,30,90,180,365,10000]

labels = [
    "0-30",
    "31-90",
    "91-180",
    "181-365",
    "365+"
]

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

In [4]:
staleness_table = (
    df.groupby("staleness_bucket")[label]
      .agg(
          n="count",
          refresh_rate="mean"
      )
)

staleness_table["refresh_rate"] *= 100
staleness_table

/tmp/ipykernel_466/2359653441.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("staleness_bucket")[label]


,n,refresh_rate
staleness_bucket,,
0-30,20476,31.837273
31-90,175,50.857143
91-180,9171,51.891833
181-365,169,12.426036
365+,5,0.0


In [5]:
trend_table = (
    df.groupby("trend_direction")[label].agg(
        n ="count",
        refresh_rate="mean"
    )
)

trend_table["refresh_rate"] *= 100
trend_table


,n,refresh_rate
trend_direction,,
down,16260,42.214022
flat,1152,0.78125
new,2236,5.366726
stable,5961,49.454789
up,4387,32.983816


-The Second Signal is : trend_direction

-Verdict : Mixed

Pages marked as new and flat rarely become refresh candidates, while down, stable, and up show much higher refresh rates. However, the expected ordering between down and stable is not observed, so the signal is only partially confirmed.

**The Rule in plain words:**

Pages receive a higher refresh score if they have not been updated recently and if their traffic trend indicates declining or stagnant performance. Recently updated or newly created pages receive a lower score because they are less likely to require immediate attention.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
df["baseline_score"] = 0
df["reason_code"] = ""

In [7]:
# Recently updated
df.loc[df["days_since_last_update"] <= 30, "baseline_score"] += 0

# Moderately stale
df.loc[
    (df["days_since_last_update"] > 30) &
    (df["days_since_last_update"] <= 180),
    "baseline_score"
] += 30

# Very stale
df.loc[df["days_since_last_update"] > 180, "baseline_score"] += 50

In [8]:
# Declining pages
df.loc[df["trend_direction"] == "down", "baseline_score"] += 40

# Stable pages
df.loc[df["trend_direction"] == "stable", "baseline_score"] += 20

# New pages
df.loc[df["trend_direction"] == "new", "baseline_score"] += 0

# Flat pages
df.loc[df["trend_direction"] == "flat", "baseline_score"] += 0

# Improving pages
df.loc[df["trend_direction"] == "up", "baseline_score"] += 10

In [9]:
df.loc[
    (df["days_since_last_update"] > 180) &
    (df["trend_direction"] == "down"),
    "reason_code"
] = "STALE_DECLINING"

df.loc[
    (df["days_since_last_update"] > 180) &
    (df["trend_direction"] == "stable"),
    "reason_code"
] = "STALE_STABLE"

df.loc[
    df["days_since_last_update"] <= 30,
    "reason_code"
] = "RECENT_CONTENT"

# fill the remaining empty values
df.loc[df["reason_code"] == "", "reason_code"] = "GENERAL_REVIEW"

In [10]:
df["action"] = "Leave As Is"

df.loc[
    df["baseline_score"] >= 70,
    "action"
] = "Refresh Now"

df.loc[
    (df["baseline_score"] >= 40) &
    (df["baseline_score"] < 70),
    "action"
] = "Review"

In [11]:
# everything ranked
ranked_df = df.sort_values(
    by="baseline_score",
    ascending=False
)

In [17]:

from pathlib import Path

output_dir = Path("/content/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

print(output_dir)

/content/work/outputs


In [18]:
# save the csv file
ranked_df.to_csv("/content/work/outputs/baseline_action_score.csv", index=False)

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
# i will review the top 10 only
top10 = ranked_df.head(10)

top10[
    [
        "content_id",
        "days_since_last_update",
        "trend_direction",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

,content_id,days_since_last_update,trend_direction,baseline_score,reason_code,action
27715,content_e454093819d5,211,down,90,STALE_DECLINING,Refresh Now
24505,content_e8e845c53339,211,down,90,STALE_DECLINING,Refresh Now
3596,content_2e2a634851a0,183,down,90,STALE_DECLINING,Refresh Now
24995,content_b57917487e5a,183,down,90,STALE_DECLINING,Refresh Now
1659,content_bbca724138f2,236,down,90,STALE_DECLINING,Refresh Now
698,content_b16bd7307b39,194,down,90,STALE_DECLINING,Refresh Now
6962,content_f01216059a6a,335,down,90,STALE_DECLINING,Refresh Now
29051,content_f85e4c6b99b8,211,down,90,STALE_DECLINING,Refresh Now
21003,content_4cc928307215,211,down,90,STALE_DECLINING,Refresh Now
7790,content_36936cb3b03c,211,down,90,STALE_DECLINING,Refresh Now


| #  | Content                | Action      | Why it's here                                       | What would make it wrong                                                     |
| -- | ---------------------- | ----------- | --------------------------------------------------- | ---------------------------------------------------------------------------- |
| 1  | `content_e454093819d5` | Refresh Now | 211 days since update + declining trend → 90 points | Could be wrong if the decline is temporary or seasonal                       |
| 2  | `content_e8e845c53339` | Refresh Now | 211 days since update + declining trend → 90 points | Could be wrong if the decline is temporary or seasonal                       |
| 3  | `content_2e2a634851a0` | Refresh Now | 183 days since update + declining trend → 90 points | Could be wrong if the decline has an external/temporary cause                |
| 4  | `content_b57917487e5a` | Refresh Now | 183 days since update + declining trend → 90 points | Could be wrong if the content is intentionally stable/seasonal               |
| 5  | `content_bbca724138f2` | Refresh Now | 236 days since update + declining trend → 90 points | Could be wrong if refreshing the page would not address the cause of decline |
| 6  | `content_b16bd7307b39` | Refresh Now | 194 days since update + declining trend → 90 points | Could be wrong if the decline is caused by something outside the content     |
| 7  | `content_f01216059a6a` | Refresh Now | 335 days since update + declining trend → 90 points | Could be wrong if the page is old but still strategically valuable           |
| 8  | `content_f85e4c6b99b8` | Refresh Now | 211 days since update + declining trend → 90 points | Could be wrong if the decline is temporary                                   |
| 9  | `content_4cc928307215` | Refresh Now | 211 days since update + declining trend → 90 points | Could be wrong if the trend does not reflect content quality                 |
| 10 | `content_36936cb3b03c` | Refresh Now | 211 days since update + declining trend → 90 points | Could be wrong if the page is intentionally seasonal                         |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

I will check the rule features against my leakage list as follows:


In [20]:
excluded_leakage = [
    "is_initial_refresh_candidate",
    "needs_indexing",
    "is_quick_win",
    "needs_ctr_fix",
    "needs_engagement_fix",
    "is_underperformer",
    "is_declining",
    "is_initial_refresh_candidate",
    "health_score",
    "ai_opportunity"
]

used_features = {
    "days_since_last_update",
    "trend_direction"
}

leakage_used = used_features.intersection(excluded_leakage)

print("Potential leakage features used:", leakage_used)

Potential leakage features used: set()


and we see there is no features from the leakage list involved in the used features

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.